In [4]:
%pip install pymongo


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.9/985.9 kB 52.8 kB/s  0:00:14 53.1 kB/s eta 0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]━━━ 1/2 [pymongo]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Cell 1 — Imports
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")

In [7]:

# Cell 2 — Load your actual saved samples from MongoDB
# Run this after you've done at least one typing test in the app
# pip install pymongo
from pymongo import MongoClient
from features import safe_extract  # reuse your features.py

client = MongoClient("mongodb://localhost:27017/neobank")
db     = client["neobank"]

# Change this to your actual userId from MongoDB
TARGET_USER_ID = "6a21102c3c6d224ed089d50e"

# Fetch samples
from bson import ObjectId
user_docs  = list(db.behaviorsamples.find({"userId": ObjectId(TARGET_USER_ID), "label": 1}))
other_docs = list(db.behaviorsamples.find({"userId": {"$ne": ObjectId(TARGET_USER_ID)}, "label": 1}))

print(f"User samples:  {len(user_docs)}")
print(f"Other samples: {len(other_docs)}")

User samples:  25
Other samples: 0


In [8]:
# Cell 3 — Extract features
X_pos = [safe_extract(d["events"]) for d in user_docs]
X_neg = [safe_extract(d["events"]) for d in other_docs]

X_pos = np.array([x for x in X_pos if x is not None])
X_neg = np.array([x for x in X_neg if x is not None])

# If no other users — use realistic placeholders
if len(X_neg) < 3:
    rng   = np.random.default_rng(42)
    X_neg = rng.uniform(50, 300, size=(10, X_pos.shape[1]))

X = np.vstack([X_pos, X_neg])
y = np.array([1] * len(X_pos) + [0] * len(X_neg))

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature vector size: {X.shape[1]}")
print(f"Class distribution — user: {sum(y==1)}, others: {sum(y==0)}")

Feature vector size: 22
Class distribution — user: 25, others: 10


In [9]:
# Cell 4 — Try different SVM parameters (find best C and gamma)
from sklearn.model_selection import GridSearchCV

svm_params = {"estimator__C": [0.1, 1, 10], "estimator__gamma": ["scale", "auto", 0.01]}
svm_grid   = GridSearchCV(
    BaggingClassifier(estimator=SVC(kernel="rbf", probability=True), n_estimators=5),
    svm_params, cv=min(3, len(X)), scoring="f1"
)
svm_grid.fit(X_scaled, y)
print("Best SVM params:", svm_grid.best_params_)
print("Best SVM score: ", svm_grid.best_score_)

Best SVM params: {'estimator__C': 0.1, 'estimator__gamma': 0.01}
Best SVM score:  1.0


In [10]:
# Cell 5 — Try different ANN hidden layer sizes
ann_params = {"estimator__hidden_layer_sizes": [(16,), (32, 16), (64, 32), (32, 16, 8)],
              "estimator__alpha":              [0.0001, 0.001]}
ann_grid   = GridSearchCV(
    BaggingClassifier(estimator=MLPClassifier(max_iter=500), n_estimators=5),
    ann_params, cv=min(3, len(X)), scoring="f1"
)
ann_grid.fit(X_scaled, y)
print("Best ANN params:", ann_grid.best_params_)
print("Best ANN score: ", ann_grid.best_score_)

Best ANN params: {'estimator__alpha': 0.0001, 'estimator__hidden_layer_sizes': (16,)}
Best ANN score:  1.0


In [11]:

# Cell 6 — Try different RF tree counts
rf_params = {"n_estimators": [20, 50, 100], "max_depth": [None, 5, 10]}
rf_grid   = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params, cv=min(3, len(X)), scoring="f1"
)
rf_grid.fit(X_scaled, y)
print("Best RF params:", rf_grid.best_params_)
print("Best RF score: ", rf_grid.best_score_)

Best RF params: {'max_depth': None, 'n_estimators': 20}
Best RF score:  1.0


In [12]:

# Cell 7 — Train final model with best params and save as .pkl
best_svm = svm_grid.best_estimator_
best_ann = ann_grid.best_estimator_
best_rf  = rf_grid.best_estimator_

best_svm.fit(X_scaled, y)
best_ann.fit(X_scaled, y)
best_rf.fit(X_scaled, y)

bundle = {"scaler": scaler, "svm": best_svm, "ann": best_ann, "rf": best_rf}
joblib.dump(bundle, f"trained_models/{TARGET_USER_ID}.pkl")
print(f"Model saved to trained_models/{TARGET_USER_ID}.pkl")

Model saved to trained_models/6a21102c3c6d224ed089d50e.pkl


In [13]:
# Cell 8 — Evaluate with cross-validation
cv = StratifiedKFold(n_splits=min(3, len(X)), shuffle=True, random_state=42)

for name, model in [("SVM", best_svm), ("ANN", best_ann), ("RF", best_rf)]:
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring="f1")
    print(f"{name} CV F1: {scores.mean():.3f} ± {scores.std():.3f}")

SVM CV F1: 1.000 ± 0.000
ANN CV F1: 1.000 ± 0.000
RF CV F1: 0.980 ± 0.028


In [14]:
# Cell 9 — Score simulation (what the live app does)
def predict_trust(feature_vector):
    fv       = np.array(feature_vector).reshape(1, -1)
    X_s      = scaler.transform(fv)
    p_svm    = best_svm.predict_proba(X_s)[0][1]
    p_ann    = best_ann.predict_proba(X_s)[0][1]
    p_rf     = best_rf.predict_proba(X_s)[0][1]
    trust    = (p_svm * 0.30 + p_ann * 0.30 + p_rf * 0.40) * 100
    action   = "allow" if trust >= 70 else "warn" if trust >= 50 else "challenge" if trust >= 30 else "block"
    print(f"Trust: {trust:.1f} | Action: {action}")
    print(f"  SVM: {p_svm*100:.1f}  ANN: {p_ann*100:.1f}  RF: {p_rf*100:.1f}")

# Test on a known-good sample (should be high trust)
if len(X_pos) > 0:
    print("Testing on own sample:")
    predict_trust(X_pos[0])

# Test on a known-bad sample (should be low trust)
if len(X_neg) > 0:
    print("Testing on other user sample:")
    predict_trust(X_neg[0])

Testing on own sample:
Trust: 98.7 | Action: allow
  SVM: 96.1  ANN: 99.5  RF: 100.0
Testing on other user sample:
Trust: 0.5 | Action: block
  SVM: 1.4  ANN: 0.3  RF: 0.0


In [15]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_predict

# Cross-validated predictions — honest estimate, not training set score
y_pred = cross_val_predict(best_rf, X_scaled, y, cv=min(3, len(X)))

acc = accuracy_score(y, y_pred)
f1  = f1_score(y, y_pred)

print(f"Accuracy: {acc:.2%}")
print(f"F1 Score: {f1:.3f}")
print(classification_report(y, y_pred, target_names=["Other user", "You"]))

Accuracy: 100.00%
F1 Score: 1.000
              precision    recall  f1-score   support

  Other user       1.00      1.00      1.00        10
         You       1.00      1.00      1.00        25

    accuracy                           1.00        35
   macro avg       1.00      1.00      1.00        35
weighted avg       1.00      1.00      1.00        35

